# Positional Encoding from Scratch

Sinusoidal and learned positional encodings in NumPy — walkthrough for `ml-learn-14-positional-encoding-scratch`.

**Equations (Vaswani et al., 2017)**

$$\mathrm{PE}(pos, 2i) = \sin\big(pos / 10000^{2i/d_{\mathrm{model}}}\big)$$

$$\mathrm{PE}(pos, 2i+1) = \cos\big(pos / 10000^{2i/d_{\mathrm{model}}}\big)$$


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if (ROOT / "positional_encoding.py").exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "positional_encoding.py").exists():
    sys.path.insert(0, str(ROOT.parent))

from positional_encoding import (
    sinusoidal_positional_encoding,
    LearnedPositionalEncoding,
    add_positional_encoding,
    position_cosine_similarity,
    check_sinusoidal_properties,
)

rng = np.random.default_rng(42)
print("imports ok")


## Sinusoidal PE matrix + heatmap

In [ ]:
T, D = 32, 64
pe = sinusoidal_positional_encoding(T, D)
print("PE shape", pe.shape)
checks = check_sinusoidal_properties(pe)
print(checks)
assert checks["matches_vaswani_formula"]

plt.figure(figsize=(9, 3.5))
plt.imshow(pe.T, aspect="auto", cmap="RdBu")
plt.colorbar()
plt.xlabel("position"); plt.ylabel("dimension")
plt.title("Sinusoidal PE (dims × positions)")
plt.show()


## Cosine similarity between positions

In [ ]:
sim = position_cosine_similarity(pe)
plt.figure(figsize=(5, 4))
plt.imshow(sim, vmin=-1, vmax=1, cmap="coolwarm")
plt.colorbar()
plt.title("Cosine similarity PE[i] · PE[j]")
plt.xlabel("j"); plt.ylabel("i")
plt.show()
print("sim(0,1)=", round(float(sim[0, 1]), 4), "  sim(0,31)=", round(float(sim[0, 31]), 4))


## Adding PE to embeddings + learned table

In [ ]:
X = rng.normal(size=(2, 16, 32))
pe16 = sinusoidal_positional_encoding(16, 32)
Xp = add_positional_encoding(X, pe16)
assert np.allclose(Xp - X, pe16)

learned = LearnedPositionalEncoding(max_len=16, d_model=32, rng=rng)
print("learned PE", learned.forward().shape)
print("add + learned ok")


## Smoke demo results

Run `python run_smoke.py` from the repo root (compares no PE / sinusoidal / learned on marker-quarter classification). Then reload the snapshot below.

In [ ]:
import json
from IPython.display import Image, display

results_dir = ROOT / "results" if (ROOT / "results").exists() else ROOT.parent / "results"
shot_path = results_dir / "JSON.shot"
if shot_path.exists():
    shot = json.loads(shot_path.read_text())
    print(json.dumps(shot, indent=2))
    for name in shot.get("plots", []):
        p = results_dir / name
        if p.exists():
            display(Image(filename=str(p)))
else:
    print("No results/ yet — run: python run_smoke.py")
